## Dataset Playground 

In [ ]:
import os, json, textwrap
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, get_dataset_config_names

sns.set_theme(style="whitegrid", font_scale=0.9)
FIG_DIR = "assets/figs"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

def parse_qa(qa):
    if isinstance(qa, dict):
        return qa
    if isinstance(qa, str):
        try:
            return json.loads(qa)
        except Exception:
            return {}
    return {}

In [ ]:
dataset_names = get_dataset_config_names("Manhph2211/PulseLM")
print(f"Configs ({len(dataset_names)}): {dataset_names}")

ALL_SPLITS = ["train", "validation", "test"]
raw = {name: {split: load_dataset("Manhph2211/PulseLM", name, split=split) for split in ALL_SPLITS} for name in dataset_names}

In [ ]:
SPLIT = "test"

In [ ]:
split_rows = []
for name in dataset_names:
    row = {"dataset": name}
    for split in ALL_SPLITS:
        row[split] = len(raw[name][split])
    row["total"] = sum(row[s] for s in ALL_SPLITS)
    split_rows.append(row)

df_splits = pd.DataFrame(split_rows).set_index("dataset")
df_splits.loc["TOTAL"] = df_splits.sum()
print(df_splits.to_string())

fig, ax = plt.subplots(figsize=(14, 5))
df_splits.drop("TOTAL")[["train", "validation", "test"]].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#4C72B0", "#DD8452", "#55A868"],
    edgecolor="white", linewidth=0.4,
)
ax.set_title("Sample counts per dataset and split")
ax.set_xlabel("")
ax.set_ylabel("samples")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
savefig("split_counts.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 10))
fig.suptitle("Amplitude distribution per dataset (all test samples)", fontsize=12, fontweight="bold")

for ax, name in zip(axes.flat, dataset_names):
    ds = raw[name][SPLIT]
    vals = np.concatenate([np.array(ds[i]["signal"], dtype=np.float32) for i in range(len(ds))])
    ax.hist(vals, bins=80, color="steelblue", edgecolor="none", density=True)
    ax.set_title(name, fontsize=8, fontweight="bold")
    ax.set_xlabel("amplitude", fontsize=7)
    ax.set_ylabel("density", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.text(0.97, 0.95, f"μ={vals.mean():.3f}\nσ={vals.std():.3f}",
            transform=ax.transAxes, fontsize=6, ha="right", va="top")

plt.tight_layout()
savefig("amplitude_dist.png")
plt.show()

In [ ]:
per_dataset_cats = defaultdict(Counter)
per_cat_answers = defaultdict(Counter)
per_dataset_per_cat_answers = defaultdict(lambda: defaultdict(Counter))
qa_counts_per_example = defaultdict(list)

for name in dataset_names:
    ds = raw[name][SPLIT]
    for i in range(len(ds)):
        qa = parse_qa(ds[i]["qa"])
        qa_counts_per_example[name].append(len(qa))
        for cat, payload in qa.items():
            ans = payload.get("answer", str(payload)) if isinstance(payload, dict) else str(payload)
            per_dataset_cats[name][cat] += 1
            per_cat_answers[cat][ans] += 1
            per_dataset_per_cat_answers[name][cat][ans] += 1

all_cats = sorted(per_cat_answers.keys())
print(f"Total QA categories: {len(all_cats)}")
for cat in all_cats:
    total = sum(per_cat_answers[cat].values())
    print(f"  {cat}: {total:,} examples  answers={dict(per_cat_answers[cat])}")

In [ ]:
heatmap_data = pd.DataFrame(
    {name: {cat: per_dataset_cats[name].get(cat, 0) for cat in all_cats} for name in dataset_names}
).T

fig, axes = plt.subplots(1, 2, figsize=(24, 6))

sns.heatmap(
    (heatmap_data > 0).astype(int), ax=axes[0],
    cmap="Blues", linewidths=0.5, linecolor="#cccccc",
    annot=True, fmt="d", cbar_kws={"label": "present"},
)
axes[0].set_title("QA category presence per dataset")
axes[0].tick_params(axis="x", rotation=40, labelsize=7)
axes[0].tick_params(axis="y", rotation=0, labelsize=8)

heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1).replace(0, 1), axis=0)
sns.heatmap(
    heatmap_norm, ax=axes[1],
    cmap="YlOrRd", linewidths=0.5, linecolor="#cccccc",
    fmt=".2f", annot=True, cbar_kws={"label": "fraction of QA pairs"},
)
axes[1].set_title("QA category fraction per dataset")
axes[1].tick_params(axis="x", rotation=40, labelsize=7)
axes[1].tick_params(axis="y", rotation=0, labelsize=8)

plt.tight_layout()
savefig("category_heatmap.png")
plt.show()

In [ ]:
global_counts = pd.Series(
    {cat: sum(per_dataset_cats[n].get(cat, 0) for n in dataset_names) for cat in all_cats}
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(global_counts.index, global_counts.values, color=sns.color_palette("tab20", len(global_counts)))
ax.set_title("Global QA category distribution (all datasets, all test examples)")
ax.set_ylabel("count")
plt.xticks(rotation=35, ha="right")
for bar, val in zip(bars, global_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(global_counts) * 0.005,
            f"{val:,}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
savefig("global_category_bar.png")
plt.show()

In [ ]:
ncols = 4
nrows = int(np.ceil(len(all_cats) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3.5))
fig.suptitle("Answer distribution per QA category (all test examples)", fontsize=13, fontweight="bold")

for ax, cat in zip(axes.flat, all_cats):
    counts = per_cat_answers[cat]
    labels, values = zip(*sorted(counts.items(), key=lambda x: -x[1]))
    bars = ax.barh(list(labels), list(values), color=sns.color_palette("pastel", len(labels)))
    ax.set_title(cat, fontsize=9, fontweight="bold")
    ax.set_xlabel("count", fontsize=8)
    ax.tick_params(labelsize=8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=7)

for ax in axes.flat[len(all_cats):]:
    ax.set_visible(False)

plt.tight_layout()
savefig("answer_dist_per_category.png")
plt.show()

In [ ]:
cats_with_answers = [c for c in all_cats if len(per_cat_answers[c]) > 1]
n_cats = len(cats_with_answers)

fig, axes = plt.subplots(n_cats, 1, figsize=(18, n_cats * 2.5))
fig.suptitle("Answer balance per QA category broken down by dataset (all test)", fontsize=12, fontweight="bold")

for ax, cat in zip(np.array(axes).flat, cats_with_answers):
    datasets_for_cat = [n for n in dataset_names if per_dataset_cats[n].get(cat, 0) > 0]
    all_answers = sorted(per_cat_answers[cat].keys())
    x = np.arange(len(datasets_for_cat))
    width = 0.8 / max(len(all_answers), 1)
    palette = sns.color_palette("Set2", len(all_answers))

    for j, ans in enumerate(all_answers):
        vals = [per_dataset_per_cat_answers[n][cat].get(ans, 0) for n in datasets_for_cat]
        ax.bar(x + j * width, vals, width, label=ans, color=palette[j])

    ax.set_title(cat, fontsize=9, fontweight="bold")
    ax.set_xticks(x + width * (len(all_answers) - 1) / 2)
    ax.set_xticklabels(datasets_for_cat, fontsize=7, rotation=25, ha="right")
    ax.set_ylabel("count", fontsize=7)
    ax.legend(fontsize=6, ncol=min(len(all_answers), 8))

plt.tight_layout()
savefig("answer_balance_by_dataset.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 10))
fig.suptitle("Text field length distribution (chars) per dataset (all test)", fontsize=12, fontweight="bold")

for ax, name in zip(axes.flat, dataset_names):
    ds = raw[name][SPLIT]
    text_lens = [len(ds[i]["text"] or "") for i in range(len(ds))]
    ax.hist(text_lens, bins=60, color="coral", edgecolor="none", density=True)
    ax.set_title(name, fontsize=8, fontweight="bold")
    ax.set_xlabel("chars", fontsize=7)
    ax.set_ylabel("density", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.text(0.97, 0.95,
            f"n={len(text_lens):,}\nμ={np.mean(text_lens):.0f}\nσ={np.std(text_lens):.0f}",
            transform=ax.transAxes, fontsize=6, ha="right", va="top")

plt.tight_layout()
savefig("text_length_dist.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 10))
fig.suptitle("Number of QA categories per example per dataset (all test)", fontsize=12, fontweight="bold")

for ax, name in zip(axes.flat, dataset_names):
    counts = qa_counts_per_example[name]
    unique = sorted(set(counts))
    ax.hist(counts, bins=max(len(unique), 1), color="mediumpurple", edgecolor="white")
    ax.set_title(name, fontsize=8, fontweight="bold")
    ax.set_xlabel("# QA categories", fontsize=7)
    ax.set_ylabel("count", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.text(0.97, 0.95,
            f"n={len(counts):,}\nmax={max(counts)}\nμ={np.mean(counts):.1f}",
            transform=ax.transAxes, fontsize=6, ha="right", va="top")

plt.tight_layout()
savefig("qa_count_per_example.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(28, 14))
fig.suptitle("PPG signal + QA/text pairs (first example per dataset)", fontsize=13, fontweight="bold", y=1.01)

for ax, name in zip(axes.flat, dataset_names):
    ds = raw[name][SPLIT]
    ex = ds[0]
    sig = np.array(ex["signal"], dtype=np.float32)
    ax.plot(sig[:1250], color="steelblue", linewidth=0.8)
    ax.set_title(name, fontsize=9, fontweight="bold")
    ax.set_xlabel("sample", fontsize=7)
    ax.set_ylabel("amplitude", fontsize=7)
    ax.tick_params(labelsize=6)

    qa = parse_qa(ex["qa"])
    text_ctx = (ex["text"] or "").strip()
    qa_lines = []
    if isinstance(qa, dict):
        for cat, payload in list(qa.items())[:4]:
            ans = payload.get("answer", payload) if isinstance(payload, dict) else payload
            qa_lines.append(f"{cat}: {ans}")

    parts = []
    if text_ctx:
        parts.append(textwrap.shorten(text_ctx, width=90, placeholder="..."))
    parts.extend(qa_lines)
    ax.text(0.01, -0.34, "\n".join(parts), transform=ax.transAxes,
            fontsize=5.5, verticalalignment="top", family="monospace")

plt.tight_layout()
savefig("ppg_grid.png")
plt.show()

In [ ]:
summary_rows = []
for name in dataset_names:
    cats_present = sorted(per_dataset_cats[name].keys())
    total_qa = sum(per_dataset_cats[name].values())
    summary_rows.append({
        "dataset": name,
        "train": len(raw[name]["train"]),
        "val": len(raw[name]["validation"]),
        "test": len(raw[name]["test"]),
        "sig_len": len(raw[name][SPLIT][0]["signal"]),
        "n_qa_cats": len(cats_present),
        "total_qa_pairs": total_qa,
        "qa_categories": ", ".join(cats_present),
    })

df_summary = pd.DataFrame(summary_rows).set_index("dataset")
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 250)
print(df_summary.to_string())
df_summary.to_csv(os.path.join(FIG_DIR, "dataset_summary.csv"))


## Benchmarking 

Hi, please read the below :)

### Main Notes: 
1) We use datasets from huggingface from now. All are preprocessed and splitted into train/val/test sets. The splits are based on subject IDs to ensure no data leakage. The exact split ratios may vary per dataset but generally follow an 80/10/10 distribution. **(ONLY report metrics using TEST SETS for the whole project.)**
2) The ppg encoders are papagei variants (papagei has 3 ckpts: "papagei_p.pt", "papagei_s.pt", "papagei_svri.pt"). We can report each of them. And each of them can be frozen or fine-tuned during training. **Let's do frozen first.**
3) Metrics extending to: strict EM-acc (old), marco_f1, rough_l, bleu1.
4) We run more experiments on: cross datasets/tasks, cross evironments, cross sensors.  

### Others:
1) It is nice to extend more LLM backbones, beside Qwen and LLAMA. Like Gemma (med), Mistral (bio). 
2) There are 2 cases when deadling with testing sets: 
    + testing on dataset-wise (like table 3 - meaning just report each set of testing sets). 
    + testing on task-wise (like table 2 - meaning merge all testing sets and report all tasks possible seperately.


In [ ]:
import subprocess, shlex, os, json

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "/home/manhph2211/workspace/PPG-Text/PE-LLM/PulseLM"
SCRIPTS  = os.path.join(NOTEBOOK_DIR, "benchmark", "scripts")
OUT_BASE = os.path.join(NOTEBOOK_DIR, "assets", "outputs")

PPG_ENCODER_TYPE = "papagei"
PPG_ENCODER_CKPT = os.path.join(NOTEBOOK_DIR, "benchmark", "checkpoints", "papagei", "papagei_s.pt")  # s, p, svri
LLM_NAME = "meta-llama/Llama-3.2-1B-Instruct"
PPG_FEAT_DIM = 512
SEED = 256

def run(cmd):
    print("$", cmd)
    subprocess.run(shlex.split(cmd), check=True)

### Case 1 — train on ALL datasets, test on ALL datasets

In [ ]:
ckpt_all = os.path.join(OUT_BASE, "train_all")

run(f"""python {SCRIPTS}/train.py
  --use_hf_dataset
  --out_dir {ckpt_all}
  --llm_name {LLM_NAME}
  --ppg_encoder_type {PPG_ENCODER_TYPE}
  --ppg_encoder_ckpt {PPG_ENCODER_CKPT}
  --ppg_feat_dim {PPG_FEAT_DIM}
  --seed {SEED}
  --per_device_train_batch_size 2
  --gradient_accumulation_steps 2
  --num_train_epochs 10""".replace("\n", " "))

In [ ]:
out_all = os.path.join(OUT_BASE, "eval_all")

run(f"""python {SCRIPTS}/test.py
  --use_hf_dataset
  --hf_split test
  --out_dir {out_all}
  --full_state_path {ckpt_all}/multimodal_full_state.pt
  --llm_name {LLM_NAME}
  --ppg_encoder_type {PPG_ENCODER_TYPE}
  --ppg_encoder_ckpt {PPG_ENCODER_CKPT}
  --ppg_feat_dim {PPG_FEAT_DIM}
  --seed {SEED}""".replace("\n", " "))

### Case 2: cross-dataset, cross-environment, cross-sensor:
+ train on vitaldb, test on: bcg, ppgbp, sensors, uci
+ train on wildppg, test on: utsappg, dalia, earset
+ train on afppgecg, test on mimicperform, ppgarrhythmia (but this ppgarrhythmia dataset has mulitple labels, so only select 2 classes: AF or Normal classes.)


In [ ]:
# Example:
TRAIN_CONFIGS = "vitaldb"

ckpt_sub = os.path.join(OUT_BASE, "train_sub")

run(f"""python {SCRIPTS}/train.py
  --use_hf_dataset
  --hf_train_names {TRAIN_CONFIGS}
  --out_dir {ckpt_sub}
  --llm_name {LLM_NAME}
  --ppg_encoder_type {PPG_ENCODER_TYPE}
  --ppg_encoder_ckpt {PPG_ENCODER_CKPT}
  --ppg_feat_dim {PPG_FEAT_DIM}
  --seed {SEED}
  --per_device_train_batch_size 8
  --gradient_accumulation_steps 8
  --num_train_epochs 10""".replace("\n", " "))

In [ ]:
out_sub = os.path.join(OUT_BASE, "eval_sub")
TEST_CONFIGS  = "bcg, ppgbp, sensors, uci"


run(f"""python {SCRIPTS}/test.py
  --use_hf_dataset
  --hf_test_names {TEST_CONFIGS}
  --hf_split test
  --out_dir {out_sub}
  --full_state_path {ckpt_sub}/multimodal_full_state.pt
  --llm_name {LLM_NAME}
  --ppg_encoder_type {PPG_ENCODER_TYPE}
  --ppg_encoder_ckpt {PPG_ENCODER_CKPT}
  --ppg_feat_dim {PPG_FEAT_DIM}
  --seed {SEED}""".replace("\n", " "))

### Report metrics 

In [ ]:
import os
import json
import pandas as pd


def report(eval_out_dir, label=""):
    pred_jsonl = os.path.join(eval_out_dir, "eval_predictions.jsonl")
    metrics_dir = os.path.join(eval_out_dir, "metrics")

    run(
        f"python {SCRIPTS}/evaluate.py "
        f"--predictions {pred_jsonl} "
        f"--out_dir {metrics_dir} "
    )

    with open(os.path.join(metrics_dir, "prediction_metrics.json")) as f:
        m = json.load(f)

    print(f"\n{'='*60}")
    if label:
        print(f"  {label}")

    print(
        f"  n={m['num_predictions']:,}  "
        f"acc={m['accuracy']:.4f}  "
        f"macro_f1={m['macro_f1']:.4f}"
    )
    print(
        f"  rouge_l={m['rouge_l']:.4f}  "
        f"bleu1={m['bleu1']:.4f}  "
        f"parse_rate={m['parse_rate']:.4f}"
    )

    print("\n  Per-category breakdown:")

    rows = []
    for cat, v in sorted(m["per_question_category"].items()):
        rows.append({
            "category": cat,
            "n": v.get("num_predictions", 0),
            "acc": round(v.get("accuracy", 0) or 0, 4),
            "macro_f1": round(v.get("macro_f1", 0) or 0, 4),
            "rouge_l": round(v.get("rouge_l", 0) or 0, 4),
            "bleu1": round(v.get("bleu1", 0) or 0, 4),
        })

    df = pd.DataFrame(rows).set_index("category")
    print(df.to_string())

    print()
    return m

In [ ]:
m_all = report(out_all, label="All-train → All-test")
m_sub = report(out_sub, label=f"Sub-train ({TRAIN_CONFIGS}) → held-out test ({TEST_CONFIGS})")